# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

*Lane: Refresh / Content Opportunity Scoring. Label: page impressions fell more than 20% (last 30d vs prior 30d). Language: observed, measured, directional, decision-support.*

## 1. Two paper findings + my methodology questions

> **Note:** I did not have the FlyRank research paper's text. If this card requires two named findings from that paper, replace the two below with them; the three questions to ask are the same. These two come from the starter repo's own documentation.

**Finding 1 (starter README): the learned model beats the hand rule roughly 3x on Precision@50 under a client-holdout split.** *Label origin:* `is_declining_label = (trend_direction == "down")`, i.e. last-30d vs prior-30d impressions. *Does validation carry the claim?* Client holdout is the right split. My questions: which features were used (any `*_last_30d` or `*_90d` totals overlap the label window), was the 30,000-page set filtered to pages with visibility (otherwise `new`/`flat` pages with zero prior impressions are trivial negatives), and what is the base rate the 3x is measured against?

**Finding 2 (data dictionary): position-tier CTR comparisons need a volume floor.** The dictionary warns that in this slice the `top_3` stratum has median ~53 impressions per 90 days, where one click moves CTR ~1.9pp, while at warehouse scale positions 1-3 run ~2.78% CTR. *Label origin:* GSC average position and CTR over 90 days. *Does validation carry the claim?* Only if the volume floor and the page count sit next to every tier number. My question: is any tier comparison in the paper reported with both?

In [3]:

import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")
URL = "https://raw.githubusercontent.com/AmaanJilani1/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)

# TARGET only. trend_direction is the label source -> never a feature.
df["y"] = (df["trend_direction"] == "down").astype(int)

# Clean past windows. The 90d total = days 1-30 (last30) + 31-60 (prev30) + 61-90 (old).
# Subtracting gives a window that is strictly BEFORE the label window.
for a, b in [("imp", "impressions"), ("clk", "clicks"), ("ses", "sessions")]:
    df[a + "_old"]  = df[b + "_90d"] - df[b + "_last_30d"] - df[b + "_prev_30d"]   # days 61-90 ago
    df[a + "_prev"] = df[b + "_prev_30d"]                                          # days 31-60 ago

# Eligibility: a page can only "decline" if it had visibility to lose.
FLOOR = 100
el = df[df["imp_prev"] >= FLOOR].copy().reset_index(drop=True)
print(f"all pages: {len(df):,} | eligible (prev30 impressions >= {FLOOR}): {len(el):,} "
      f"| clients: {el.client_id.nunique()} | decline base rate: {el.y.mean():.3f}")


from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Raw columns the features are built from. Everything is from BEFORE the label window.
SOURCES = ["imp_old","imp_prev","clk_old","clk_prev","ses_old","ses_prev","search_volume","competition",
           "cpc","word_count","content_age_days","days_since_last_update","content_type","main_intent"]
FORBIDDEN = {"trend_direction","trend_pct","impressions_last_30d","clicks_last_30d","sessions_last_30d",
             "impressions_90d","clicks_90d","sessions_90d","ctr","avg_position","engagement_rate",
             "scroll_rate","ai_traffic_pct","impression_tier","position_tier","y"}
assert not (set(SOURCES) & FORBIDDEN), "a forbidden column is feeding the features"

def make_features(d):
    X = pd.DataFrame(index=d.index)
    for a in ["imp", "clk", "ses"]:
        X["log_" + a + "_old"]  = np.log1p(d[a + "_old"])
        X["log_" + a + "_prev"] = np.log1p(d[a + "_prev"])
        X[a + "_momentum"] = X["log_" + a + "_prev"] - X["log_" + a + "_old"]   # >0 growing, <0 falling
    X["ctr_old"]  = d.clk_old  / d.imp_old.clip(lower=1)
    X["ctr_prev"] = d.clk_prev / d.imp_prev.clip(lower=1)
    X["ctr_delta"] = X.ctr_prev - X.ctr_old
    X["log_search_volume"] = np.log1p(d.search_volume)
    X["competition"], X["cpc"] = d.competition, d.cpc
    X["word_count"], X["wc_missing"] = d.word_count, d.word_count.isna().astype(int)
    X["content_age_days"] = d.content_age_days
    X["updated_over_30d_ago"] = (d.days_since_last_update > 30).astype(int)  # 0 = updated INSIDE the label window
    X["is_keyword_article"]  = (d.content_type == "keyword article").astype(int)
    X["is_comparison"]       = (d.content_type == "comparison article").astype(int)
    for v in ["informational", "transactional", "commercial", "navigational"]:
        X["intent_" + v] = (d.main_intent == v).astype(int)
    return X

X, y, g = make_features(el), el["y"].values, el["client_id"].values

def precision_at_k(y, s, k):
    return y[np.argsort(-s, kind="stable")[:k]].mean()

def cv_scores(model_fn, X, y, g, split="group", n=5, seed=0):
    """Out-of-fold probabilities. split='group' = client holdout (honest); 'random' = leaky comparison."""
    oof = np.zeros(len(y))
    sp = GroupKFold(n).split(X, y, g) if split == "group" else StratifiedKFold(n, shuffle=True, random_state=seed).split(X, y)
    for tr, te in sp:
        m = model_fn(); m.fit(X.iloc[tr], y[tr]); oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof

make_lr  = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))
make_hgb = lambda: HGB(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0)

def report(name, s, y=y):
    return {"model": name, "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
            "P@50": precision_at_k(y, s, 50), "P@100": precision_at_k(y, s, 100), "P@500": precision_at_k(y, s, 500)}

# Baselines (no fitting): A = the original ML-07 rule, B = "last period's drop continues"
score_rule = ((el.days_since_last_update >= 180) & (el.impressions_90d >= 500)).astype(float).values * el.impressions_90d.values
score_persist = (-X["imp_momentum"]).values

# Does my label really equal the dictionary rule, and what does eligibility change?
prev, last = df.impressions_prev_30d, df.impressions_last_30d
chg = (last - prev) / prev * 100
rule = np.where((prev == 0) & (last > 0), "new", np.where((prev == 0), "flat", np.where(chg > 20, "up", np.where(chg < -20, "down", "stable"))))
print("label reproduces trend_direction exactly:", (rule == df.trend_direction).all())
print(f"label base rate: all pages {df.y.mean():.3f} vs eligible pages {y.mean():.3f}")
print("share of all pages that can never be 'down' (prev30 = 0):", round((prev == 0).mean(), 3))


all pages: 30,000 | eligible (prev30 impressions >= 100): 18,010 | clients: 30 | decline base rate: 0.616
label reproduces trend_direction exactly: True
label base rate: all pages 0.542 vs eligible pages 0.616
share of all pages that can never be 'down' (prev30 = 0): 0.113


## 2. My model under an honest split (before/after)

**Before (random 5-fold, same page-level split most notebooks default to):** ROC-AUC 0.68 (logistic) and 0.72 (boosting). **After (client holdout):** 0.61 and 0.62. The gap is the cost of client leakage: when pages from the same client sit on both sides of the split, the model partly learns *the client*. The honest number is the lower one.

In [4]:

rows = []
for nm, fn in [("Logistic regression", make_lr), ("Gradient boosting", make_hgb)]:
    for split in ["random", "group"]:
        r = report(f"{nm} [{split}]", cv_scores(fn, X, y, g, split)); r["split"] = split; rows.append(r)
res = pd.DataFrame(rows).round(3); print(f"base rate {y.mean():.3f}"); print(res.drop(columns="split").to_string(index=False))


base rate 0.616
                       model  ROC-AUC  PR-AUC  P@50  P@100  P@500
Logistic regression [random]    0.682   0.762  0.94   0.88  0.882
 Logistic regression [group]    0.615   0.702  0.88   0.85  0.804
  Gradient boosting [random]    0.716   0.796  0.92   0.92  0.926
   Gradient boosting [group]    0.609   0.689  0.74   0.72  0.702


## 3. Leakage audit

Three checks on the final feature set. (1) **Source audit:** no label-window or target columns feed the features (assertion). (2) **Deliberate leak:** I add `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` and `impressions_90d` and AUC jumps to ~0.998. That is what leakage looks like, and why those columns are banned. (3) **Shuffle-label test:** with labels shuffled the clean model should fall to chance (~0.5).

In [5]:

assert not (set(SOURCES) & FORBIDDEN)
Xl = X.copy()
for c in ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]: Xl[c] = el[c]
Xl["impressions_90d"] = el.impressions_90d
leaky = roc_auc_score(y, cv_scores(make_hgb, Xl, y, g, "group"))
ys = np.random.RandomState(1).permutation(y)
shuf = roc_auc_score(ys, cv_scores(make_hgb, X, ys, g, "group"))
clean = roc_auc_score(y, cv_scores(make_hgb, X, y, g, "group"))
print(f"clean features, real labels      AUC {clean:.3f}")
print(f"LEAKY (label-window cols added)  AUC {leaky:.3f}   <- never ship this")
print(f"clean features, shuffled labels  AUC {shuf:.3f}   <- should be ~0.50")
print(f"\nCaveat: `updated_over_30d_ago` flags pages edited inside the label window (updated_over_30d_ago = 0 for {(X.updated_over_30d_ago == 0).mean():.0%} of pages). "
      f"Their decline rate is {y[X.updated_over_30d_ago == 0].mean():.3f} vs {y[X.updated_over_30d_ago == 1].mean():.3f}, so this is a possible confound and I report it, not model on it as a cause.")


clean features, real labels      AUC 0.609
LEAKY (label-window cols added)  AUC 0.997   <- never ship this
clean features, shuffled labels  AUC 0.499   <- should be ~0.50

Caveat: `updated_over_30d_ago` flags pages edited inside the label window (updated_over_30d_ago = 0 for 61% of pages). Their decline rate is 0.608 vs 0.628, so this is a possible confound and I report it, not model on it as a cause.


## 4. Claim rewrite

**My boldest sentence (before):** *"The model predicts which pages will decline, so refreshing them recovers traffic."*

**Rewrite (after):** *"On 18,010 pages from 30 anonymized clients, a logistic model using only pre-label-window signals ranked pages that later lost more than 20% of impressions better than a persistence baseline when scored on clients it had not seen (ROC-AUC 0.61 vs 0.57; precision@100 0.85 vs 0.64, base rate 0.62). This is an observed, directional association that supports prioritizing human review. It does not show that refreshing a page changes its traffic, and it says nothing about how Google ranks pages."*

In [6]:

lr_g = report("lr", cv_scores(make_lr, X, y, g, "group")); pb = report("b", score_persist)
print(f"On {len(y):,} pages / {len(set(g))} clients: logistic AUC {lr_g['ROC-AUC']:.2f} vs persistence {pb['ROC-AUC']:.2f}; "
      f"P@100 {lr_g['P@100']:.2f} vs {pb['P@100']:.2f}; base {y.mean():.2f}.  Observed / directional / decision-support only.")


On 18,010 pages / 30 clients: logistic AUC 0.62 vs persistence 0.57; P@100 0.85 vs 0.64; base 0.62.  Observed / directional / decision-support only.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.